In [1]:
import pandas as pd
import numpy as np
import os

def calculate_gini(array):
    # Calculating Gini coefficient of a numpy array.
    # To handle the problem of negative values, we will clip values below 0
    # A player with negative WAR is essentially providing 0 wins over a replacement-level player.

    array = np.array(array, dtype=np.float64)
    array = np.clip(array, 0, None) # Convert negative into 0 to prevent Gini formula from breaking
        
    if np.sum(array) == 0:
        return 0.0
        
    array = np.sort(array)
    index = np.arange(1, array.shape[0] + 1)
    n = array.shape[0]
    return ((np.sum((2 * index - n - 1) * array)) / (n * np.sum(array)))

print("Loading raw data...")
df_bat = pd.read_csv('../data/raw_data/raw_batters_2010_2025.csv')
df_pit = pd.read_csv('../data/raw_data/raw_pitchers_2010_2025.csv')

# 1. Remove 'TOT' rows to prevent double counting traded players
df_bat = df_bat[df_bat['Team'] != 'TOT'].copy()
df_pit = df_pit[df_pit['Team'] != 'TOT'].copy()

# 2. Classify Pitchers: Starters (GS > G/2) vs Relievers (GS <= G/2)
# If a pitcher starts more than half of their appearances, they are a Starter
df_pit['Position'] = np.where(df_pit['GS'] > (df_pit['G'] / 2), 'SP', 'RP')
df_bat['Position'] = 'Batter'

# Combine batters and pitchers into one DataFrame
df_all = pd.concat([df_bat, df_pit], ignore_index=True)

# 3. Calculate Team-Year Features
results = []
grouped = df_all.groupby(['Season', 'Team'])

print("Calculating Gini Coefficients and WAR distributions per team...")
for (season, team), group in grouped:
    total_war = group['WAR'].sum()
    
    # Calculate WAR by Position
    bat_war = group[group['Position'] == 'Batter']['WAR'].sum()
    sp_war = group[group['Position'] == 'SP']['WAR'].sum()
    rp_war = group[group['Position'] == 'RP']['WAR'].sum()
    
    # Calculate Top 3 Player WAR Reliance 
    top_3_war = group['WAR'].nlargest(3).sum()
    top_3_ratio = top_3_war / total_war if total_war > 0 else 0
    
    # Calculate Gini Coefficient
    gini_coef = calculate_gini(group['WAR'].values)
    
    results.append({
        'Season': season,
        'Team': team,
        'Total_WAR': total_war,
        'Batter_WAR': bat_war,
        'SP_WAR': sp_war,
        'RP_WAR': rp_war,
        'Top3_WAR_Ratio': top_3_ratio,
        'Gini_Coefficient': gini_coef
    })

df_features = pd.DataFrame(results)

# 4. Save to Processed Data
os.makedirs('../data/processed_data', exist_ok=True)
save_path = '../data/processed_data/team_war_features_2010_2025.csv'
df_features.to_csv(save_path, index=False)

print(f"Feature engineering complete! Data saved to: {save_path}")
print(df_features.head())

Loading raw data...
Calculating Gini Coefficients and WAR distributions per team...
Feature engineering complete! Data saved to: ../data/processed_data/team_war_features_2010_2025.csv
   Season   Team  Total_WAR  Batter_WAR  SP_WAR  RP_WAR  Top3_WAR_Ratio  \
0    2010  - - -       44.2        13.7    29.2     1.3        0.348416   
1    2010    ARI       17.8        19.8     1.0    -3.0        0.803371   
2    2010    ATL       35.7        20.1     9.8     5.8        0.406162   
3    2010    BAL       19.5         9.9     7.3     2.3        0.420513   
4    2010    BOS       42.9        26.4    15.5     1.0        0.361305   

   Gini_Coefficient  
0          0.811518  
1          0.876249  
2          0.749259  
3          0.790389  
4          0.794088  


In [7]:
import pandas as pd
import numpy as np
import os

print("Loading engineered WAR features...")
# Load the CSV you just saved from the previous step
df_features = pd.read_csv('../data/processed_data/team_war_features_2010_2025.csv')
df_features = df_features[df_features['Season'] <= 2024].copy()

print("Loading Kaggle payroll data for Wins/Losses...")
# Updated path to match your new directory structure
df_standings = pd.read_csv('../data/raw_data/mlb_payrolls.csv')

# Map Kaggle abbreviations to FanGraphs abbreviations
abbr_correction = {
    'TB': 'TBR', 'WSH': 'WSN', 'CWS': 'CHW', 
    'KC': 'KCR', 'SD': 'SDP', 'SF': 'SFG'
}
df_standings['Team'] = df_standings['Team'].replace(abbr_correction)

# Extract only the columns we need
df_standings = df_standings[['Year', 'Team', 'Wins', 'Losses']]

print("Merging standings data...")
# Merge using Season(FanGraphs) and Year(Kaggle)
df_final = pd.merge(df_features, df_standings, left_on=['Season', 'Team'], right_on=['Year', 'Team'], how='inner')

# Hardcoded World Series Winners (2011 - 2024, since Kaggle data starts at 2011)
ws_winners = {
    2011: 'STL', 2012: 'SFG', 2013: 'BOS', 2014: 'SFG',
    2015: 'KCR', 2016: 'CHC', 2017: 'HOU', 2018: 'BOS', 2019: 'WSN',
    2020: 'LAD', 2021: 'ATL', 2022: 'HOU', 2023: 'TEX', 2024: 'LAD'
}

print("Labeling World Series Champions...")
df_final['WS_Winner'] = df_final.apply(
    lambda row: 1 if ws_winners.get(row['Season']) == row['Team'] else 0, axis=1
)

df_final['Win_PCT'] = df_final['Wins'] / (df_final['Wins'] + df_final['Losses'])

# Drop redundant Kaggle 'Year' column
df_final = df_final.drop(columns=['Year'])

final_save_path = '../data/processed_data/final_team_data.csv'
df_final.to_csv(final_save_path, index=False)

print(f"Data merge complete! Final dataset saved to: {final_save_path}")
print(df_final[['Season', 'Team', 'Wins', 'WS_Winner', 'Gini_Coefficient']].head())

Loading engineered WAR features...
Loading Kaggle payroll data for Wins/Losses...
Merging standings data...
Labeling World Series Champions...
Data merge complete! Final dataset saved to: ../data/processed_data/final_team_data.csv
   Season Team  Wins  WS_Winner  Gini_Coefficient
0    2011  ARI    94          0          0.826056
1    2011  ATL    89          0          0.757849
2    2011  BAL    69          0          0.865181
3    2011  BOS    90          0          0.841021
4    2011  CHC    71          0          0.805195
